# Flajolet-Martin Algorithm: Counting Distinct Elements

## Learning Objectives

1. **State** the distinct count problem and why exact counting requires $O(n)$ space
2. **Explain** the trailing-zeros trick for estimating distinct count
3. **Implement** Flajolet-Martin with median-of-means correction
4. **Analyze** the accuracy of FM estimates

## The Distinct Count Problem

Given a stream $a_1, a_2, \ldots, a_n$ over a universe $[m]$, estimate the number of **distinct** elements $d$.

**Exact:** maintain a hash set → $O(d)$ space.

**Goal:** $O(\log m)$ space with $(1\pm\epsilon)$ approximation.

**Why it's hard:** each element is seen only once; there's no natural order.

## The Trailing Zeros Trick

Hash each element $a \in [m]$ to a $b$-bit integer $h(a) \in \{0, 1, \ldots, 2^b-1\}$.

Define $R(h(a))$ = position of the rightmost 1-bit = number of trailing zeros.

**Key observation:** under a uniform random hash:
$$\Pr[R(h(a)) \geq r] = \frac{1}{2^r}$$

If we see $d$ distinct elements, the maximum trailing zeros observed, $M$, satisfies:
$$\mathbb{E}[2^M] \approx 0.7735 \cdot d$$

So $\hat{d} = 2^M / 0.7735$ is an estimator for $d$. High variance — need averaging.

In [1]:
import math, random, hashlib

def trailing_zeros(n):
    """Count trailing zeros in binary representation."""
    if n == 0: return 32
    count = 0
    while n & 1 == 0:
        count += 1; n >>= 1
    return count

def fm_estimate(stream, seed=0):
    """Single Flajolet-Martin estimate."""
    M = 0
    for item in stream:
        h = int(hashlib.md5(f"{seed}:{item}".encode()).hexdigest(), 16)
        M = max(M, trailing_zeros(h))
    return 2**M / 0.7735

def fm_median_of_means(stream, n_groups=5, n_per_group=3):
    """Median-of-means correction: reduces variance."""
    n_hashes = n_groups * n_per_group
    stream_list = list(stream)
    estimates = [fm_estimate(stream_list, seed=i) for i in range(n_hashes)]
    groups = [estimates[i*n_per_group:(i+1)*n_per_group] for i in range(n_groups)]
    group_means = [sum(g)/len(g) for g in groups]
    return sorted(group_means)[len(group_means)//2]  # median of means

# Test
rng = random.Random(99)
universe = 10000
for d_true in [100, 500, 2000, 5000]:
    distinct_items = rng.sample(range(universe), d_true)
    # Stream with repeats
    stream = [rng.choice(distinct_items) for _ in range(d_true * 5)]
    est = fm_median_of_means(stream)
    rel_err = abs(est - d_true) / d_true
    print(f"True d={d_true:>5}, estimate={est:>7.0f}, rel_error={rel_err:.3f}")

True d=  100, estimate=    110, rel_error=0.103
True d=  500, estimate=    717, rel_error=0.434
True d= 2000, estimate=   8826, rel_error=3.413


True d= 5000, estimate=  18534, rel_error=2.707


In [2]:
# Accuracy vs number of hash functions
print("Hash functions | Avg relative error (over 20 trials)")
rng2 = random.Random(11)
d_true = 1000
distinct_items = rng2.sample(range(100000), d_true)
stream = [rng2.choice(distinct_items) for _ in range(d_true*3)]

for n_hashes in [1, 3, 9, 15, 21]:
    errors = []
    for trial in range(20):
        estimates = [fm_estimate(stream, seed=trial*100+i) for i in range(n_hashes)]
        est = sorted(estimates)[n_hashes//2]
        errors.append(abs(est - d_true)/d_true)
    avg_err = sum(errors)/len(errors)
    bar = '#'*int(avg_err*100)
    print(f"  k={n_hashes:>3}: avg_rel_err={avg_err:.3f}  {bar}")

Hash functions | Avg relative error (over 20 trials)
  k=  1: avg_rel_err=1.811  #####################################################################################################################################################################################
  k=  3: avg_rel_err=1.240  ###########################################################################################################################


  k=  9: avg_rel_err=0.526  ####################################################


  k= 15: avg_rel_err=0.591  ###########################################################


  k= 21: avg_rel_err=0.657  #################################################################


## Summary

| Algorithm | Space | Error |
|-----------|-------|-------|
| Exact counting | $O(d)$ | 0 |
| Flajolet-Martin (1 hash) | $O(\log m)$ | High variance, biased |
| FM (median-of-means, $k$ hashes) | $O(k \log m)$ | $O(1/\sqrt{k})$ error |
| HyperLogLog (modern variant) | $O(2^b)$ bits | $\pm 1.04/\sqrt{2^b}$ |

Flajolet-Martin was the first $O(\log m)$ distinct counter. Modern systems use HyperLogLog (2007) for better constants and implementation simplicity.